In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path
import random

In [ ]:
# NOTE: aexp0.2 is finetuned ResNet
#       aexp0.3 is pretrained ResNet

# read results
attack_dfs = []
out_dir = Path('../results/aexp0.2')
for attack_df_path in out_dir.glob('results/*.csv'):
    attack_df = pd.read_csv(attack_df_path, index_col=0)
    attack_dfs.append(attack_df)
df = pd.concat(attack_dfs, axis=0)
df.index = range(len(df))

# add zero occlusion result to each attack type for easier graphing
oa_name_to_hr = {
    'none': 'Unoccluded',
    'rand_simplex_noise_1': 'Random (simplex noise)',
    'rand_patch_8': 'Random (patch)',
    'grad_cam': 'Grad-CAM',
    'diff_ev_8_0.5_32_100': 'DE',
}
oa_names = list(oa_name_to_hr.keys())
for oa_name in oa_names:
    if oa_name == 'none':
        continue
    new_rows = df[df.oa_name == 'none'].copy()
    new_rows['oa_name'] = oa_name
    noise_freq = df[df.oa_name == oa_name].noise_freq.unique().item()
    new_rows['noise_freq'] = noise_freq
    df = pd.concat([df, new_rows], axis=0)

fns = sorted(df.filename.unique())
occ_fracs = sorted(df.occ_frac.unique())
pos_occ_fracs = occ_fracs[1:]

# subset df to include only wanted trials
df = df[df.oa_name.isin(oa_names)]
df['oa_hr_name'] = df.oa_name.map(lambda e: oa_name_to_hr[e])

print(len(df))
df.groupby('oa_name').correct.count()

In [ ]:
def oa_acc_vs_occ_frac(df, name, save_name=None):
    # plot to compare different attacks
    fig, ax = plt.subplots(tight_layout=True)
    df = df[df.oa_name != 'none']
    sns.lineplot(
        data=df, x='occ_frac', y='correct', hue='oa_hr_name', marker='o', 
        errorbar='ci', err_style='bars', ax=ax, palette=sns.palettes.color_palette('Set1'),
    )
    ax.set_xticks(occ_fracs)
    ax.set_xticklabels(occ_fracs)
    ax.legend(title='', loc='lower left')
    plt.title(f'Accuracy vs. occlusion fraction by attack ({name})')
    plt.legend(title='Attack')
    plt.ylabel('Accuracy')
    plt.xlabel('Occlusion fraction')
    if save_name is None:
        save_name = name
    plt.savefig(f'out/attack_acc_vs_occ_frac_{save_name}.png')
    plt.show()
    
oa_acc_vs_occ_frac(df=df, name='finetuned ResNet-50', save_name='finetuned_resnet50')

In [ ]:
# plot
classes_path = Path('../data/imagenet/ann/classes.csv')
classes_df = pd.read_csv(classes_path, index_col=0)

def attack_ex_ims(df, name, save_name=None):
    random.seed(1)
    ncol = len(oa_names)
    nrow = len(pos_occ_fracs)
    fig, axss = plt.subplots(nrow, ncol, constrained_layout=True)
    fig.set_size_inches(2 * ncol, 3 * nrow)
    fig.suptitle(f'Attack image examples ({name})', fontsize=14)
    for row, (axs, occ_frac) in enumerate(zip(axss, pos_occ_fracs)):
        fn = random.choice(fns)
        for col, (ax, oa_name) in enumerate(zip(axs, oa_names)):
            if oa_name == 'none':
                # unoccluded case
                oa_full_name = f'{oa_name}_0'
                s = df[(df.oa_name == oa_name) & (df.filename == fn) & (df.occ_frac == 0)].squeeze()
            else:
                oa_full_name = f'{oa_name}_{occ_frac}'
                s = df[(df.oa_name == oa_name) & (df.filename == fn) & (df.occ_frac == occ_frac)].squeeze()
            im = Image.open(out_dir / oa_full_name / fn)

            targ_prob = s.targ_prob
            best_nontarg_prob = s.best_nontarg_prob
            targ_label = classes_df.loc[s.gt_idx].label
            best_nontarg_label = classes_df.loc[s.best_nontarg_idx].label

            xlabel = f'P({targ_label}) = {targ_prob:.3f}'
            xlabel += f'\nP({best_nontarg_label}) = {best_nontarg_prob:.3f}'
            ax.set_xlabel(xlabel)

            if row == 0:
                oa_hr_name = oa_name_to_hr[oa_name]
                ax.set_title(oa_hr_name)
            if col == 0:
                ax.set_ylabel(f'f = {occ_frac}')

            ax.imshow(im)

            ax.set_xticks([])
            ax.set_yticks([])
            for e in ['top', 'right', 'bottom', 'left']:
                ax.spines[e].set_visible(False)
    if save_name is None:
        save_name = name
    plt.savefig(f'out/attack_ex_ims_{save_name}.png')
    
attack_ex_ims(df=df, name='finetuned ResNet-50', save_name='finetuned_resnet50')